# The Complete PyTorch Course (Beginner → Intermediate → Advanced)

This notebook is a compact but comprehensive, end-to-end PyTorch course designed to take you from **zero** to **production-grade** workflows.

It is organized as three courses in one:

- **Beginner Course:** tensors, autograd, `nn.Module`, training loops, `Dataset`/`DataLoader`, saving/loading
- **Intermediate Course:** better training (regularization, schedulers, mixed precision), debugging/profiling, common model families (CNN/RNN/Transformer)
- **Advanced Course:** performance & compilation (`torch.compile`), exporting (`torch.export`), quantization, distributed training (DDP/FSDP), deployment patterns, tokenizers & text pipelines

## Assumptions
- You know basic Python.
- You do **not** need prior deep learning experience.

## Notes on “tokenizers with torch”
Core PyTorch is primarily a tensor + autograd + model runtime library; **tokenization** is usually handled by:
- your own preprocessing code (often enough for research and many products), or
- specialized libraries (e.g., Hugging Face `tokenizers/transformers`).

This course includes:
- **pure-Python + PyTorch** tokenization and numericalization pipelines (word/char/BPE)
- a **minimal BPE tokenizer** implementation for learning and small projects
- patterns to integrate external tokenizers with PyTorch `Dataset`/`DataLoader` when you need industrial tokenizers

---

## Table of Contents
1. Setup & Environment
2. Beginner Course
   - Tensors & tensor semantics
   - Autograd fundamentals
   - `nn.Module` and parameters
   - Training loops (from scratch)
   - Datasets and DataLoaders
   - Saving/loading & inference mode
3. Intermediate Course
   - Optimization best practices
   - Regularization & initialization
   - Schedulers, gradient clipping/accumulation
   - Mixed precision (AMP)
   - Debugging: hooks, anomaly detection
   - Profiling & performance hygiene
   - CNNs and common vision patterns
   - NLP basics: embeddings, padding, masking
4. Advanced Course
   - `torch.compile` (PyTorch 2.x compiler stack)
   - `torch.export` and deployment graphs
   - ONNX export overview
   - Quantization (PTQ/QAT)
   - Distributed training (DDP/FSDP) and scaling
   - Large-model training tactics
   - Deployment patterns (server + edge)

---

**Tip:** Run the notebook top-to-bottom once. Then revisit sections as a reference.

## 1) Setup & Environment

### Install PyTorch
Use the official installer for your OS + CUDA/ROCm/CPU combination.

Typical pip installs (examples):
```bash
# CPU-only (simple default)
pip install torch torchvision torchaudio

# CUDA wheels (example index)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
```

### Verify installation
The next cell prints your Torch version and whether CUDA is available.

In [ ]:

import torch, platform
print("PyTorch:", torch.__version__)
print("Python :", platform.python_version())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

### Reproducibility quick-start
Full determinism can reduce performance and isn't always possible for every op, but for learning and debugging it's helpful.

In [ ]:

import torch, random
import numpy as np

def seed_all(seed: int = 1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinism (can slow things down; may raise errors for non-deterministic ops)
    torch.use_deterministic_algorithms(False)

seed_all(42)

# 2) Beginner Course

## 2.1 Tensors: the core data structure

A **tensor** is an n-dimensional array with:
- a **dtype** (float32, int64, bfloat16, …)
- a **device** (CPU, CUDA GPU, MPS on Apple Silicon, …)
- a **shape** (e.g., `[batch, channels, height, width]`)

Key idea: **PyTorch is eager** by default — ops execute immediately, returning new tensors.

In [ ]:

import torch

# Creation
x = torch.tensor([[1,2,3],[4,5,6]], dtype=torch.float32)
y = torch.zeros((2,3))
z = torch.randn((2,3))

print("x:", x)
print("dtype:", x.dtype, "shape:", x.shape, "device:", x.device)

# Common gotcha (from: torch.tensor([..],[..]) error)
# Correct: pass a single nested list to make a 2D tensor:
a = torch.tensor([[1,2,3],[4,5,6]])
print("a shape:", a.shape)

### Devices
Use `.to(device)` to move tensors and models. Prefer choosing a device once and reusing it.

In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
x = torch.randn(3, 4).to(device)
print(device, x.device)

### Shapes, views, and memory
- `view` requires contiguous memory; `reshape` can copy if needed.
- `.contiguous()` forces contiguous layout.

In [ ]:

t = torch.arange(12).reshape(3,4)
tT = t.t()  # transpose produces a non-contiguous view
print("contiguous?", tT.is_contiguous())
try:
    tT.view(-1)
except RuntimeError as e:
    print("view failed:", e)

flat = tT.reshape(-1)  # reshape handles non-contiguous by copying if required
print("reshape ok:", flat.shape)

### Broadcasting
Broadcasting lets you combine tensors of compatible shapes without manually repeating data.
Rule of thumb: align shapes from the **right**, and dimensions must be equal or 1.

In [ ]:

A = torch.randn(10, 1)
B = torch.randn(1, 20)
C = A + B  # (10,1) + (1,20) -> (10,20)
print(C.shape)

### Indexing & slicing
PyTorch supports:
- basic slicing (`:`)
- integer indexing
- boolean masks
- advanced indexing with index tensors

In [ ]:

x = torch.arange(0, 24).reshape(2,3,4)
print("x[0]:\n", x[0])
print("x[:, :, 1]:\n", x[:,:,1])

mask = x > 10
print("masked values:", x[mask][:10], "...")

idx = torch.tensor([0,2])
print("take columns 0 and 2 from 2nd dim:\n", x[:, idx, :].shape)

## 2.2 Autograd: automatic differentiation

PyTorch builds a computation graph dynamically. When `requires_grad=True`, PyTorch tracks operations to compute gradients via `.backward()`.

Key rules:
- Gradients accumulate by default into `.grad`.
- Call `optimizer.zero_grad(set_to_none=True)` or `param.grad = None` each step.
- Use `torch.no_grad()` or `torch.inference_mode()` for inference.

In [ ]:

x = torch.tensor(2.0, requires_grad=True)
y = 3*x**2 + 2*x + 1
y.backward()
print("dy/dx at x=2:", x.grad)  # 3*2*(2) + 2 = 14

### Common gradient controls
- `.detach()` creates a tensor that shares storage but is not tracked by autograd.
- `with torch.no_grad():` disables tracking inside the block.
- `torch.inference_mode()` is even more optimized for inference.

In [ ]:

x = torch.randn(3, requires_grad=True)
y = (x * 2).sum()
y.backward()
print("grad:", x.grad)

x2 = x.detach()
print("x2 requires_grad?", x2.requires_grad)

with torch.no_grad():
    z = x * 3
print("z requires_grad?", z.requires_grad)

with torch.inference_mode():
    w = x * 4
print("w requires_grad?", w.requires_grad)

## 2.3 `nn.Module`: building blocks for models

A model is a Python class inheriting from `nn.Module`.
- Layers are assigned as attributes.
- The forward pass is implemented in `forward`.
- Parameters are automatically registered.

You can inspect:
- `model.parameters()` for trainable tensors
- `model.state_dict()` for serialization

In [ ]:

import torch.nn as nn
import torch.nn.functional as F

class TinyMLP(nn.Module):
    def __init__(self, in_dim=10, hidden=32, out_dim=2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

m = TinyMLP()
print(m)
print("num params:", sum(p.numel() for p in m.parameters()))

### A note on `torch.nn.functional` vs `nn.Module`
- Use `nn.Module` layers for anything with parameters or buffers (e.g. `nn.Linear`, `nn.BatchNorm2d`).
- Use `torch.nn.functional` for stateless ops (e.g. `F.relu`, `F.softmax`).

## 2.4 Training loop: from scratch

The canonical training loop:
1. forward pass
2. compute loss
3. `loss.backward()`
4. `optimizer.step()`
5. clear grads

We'll train a regression model on synthetic data.

In [ ]:

import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Synthetic regression: y = Wx + b + noise
N, D = 2048, 5
true_W = torch.randn(D, 1)
true_b = torch.randn(1)

X = torch.randn(N, D)
y = X @ true_W + true_b + 0.1*torch.randn(N,1)

X, y = X.to(device), y.to(device)

model = nn.Linear(D, 1).to(device)
opt = optim.Adam(model.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

for step in range(300):
    pred = model(X)
    loss = loss_fn(pred, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 50 == 0:
        print(f"step={step:03d} loss={loss.item():.6f}")

print("Learned W (first 3):", model.weight.detach().cpu().view(-1)[:3])

## 2.5 Datasets and DataLoaders

For real training you rarely load everything into memory. PyTorch provides:
- `torch.utils.data.Dataset`: define how to index examples
- `torch.utils.data.DataLoader`: batching, shuffling, multiprocessing

We'll build a tiny dataset and train using mini-batches.

In [ ]:

from torch.utils.data import Dataset, DataLoader

class RegressionDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return self.X.shape[0]
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

ds = RegressionDataset(X, y)
dl = DataLoader(ds, batch_size=64, shuffle=True, num_workers=0)

model = nn.Linear(D, 1).to(device)
opt = optim.SGD(model.parameters(), lr=5e-2)
loss_fn = nn.MSELoss()

for epoch in range(5):
    total = 0.0
    for xb, yb in dl:
        pred = model(xb)
        loss = loss_fn(pred, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    print(f"epoch={epoch} loss={total/len(ds):.6f}")

## 2.6 Saving and loading models

Two common patterns:
1. Save `state_dict` (recommended)
2. Save entire model object (less portable; depends on class code)

Always consider saving:
- model state
- optimizer state
- scheduler state
- AMP scaler state (if using AMP)
- RNG states (optional)

In [ ]:

import os, torch
ckpt_path = "pytorch_course_checkpoint.pt"

checkpoint = {
    "model_state": model.state_dict(),
    "opt_state": opt.state_dict(),
    "meta": {"D": D}
}
torch.save(checkpoint, ckpt_path)

# Load
loaded = torch.load(ckpt_path, map_location=device)
model2 = nn.Linear(loaded["meta"]["D"], 1).to(device)
model2.load_state_dict(loaded["model_state"])

print("Loaded ok:", all((a==b).all().item() for a,b in zip(model.parameters(), model2.parameters())))

---

# 3) Intermediate Course

The intermediate course focuses on training quality, stability, and real-world workflows.

## 3.1 Optimization: losses, optimizers, and schedules

### Loss selection
- Regression: `MSELoss`, `SmoothL1Loss` (Huber)
- Classification: `CrossEntropyLoss` (logits in, labels out)
- Multi-label: `BCEWithLogitsLoss`
- Metric learning: contrastive / triplet losses

### Optimizer selection (rules of thumb)
- **AdamW**: strong default for Transformers and many modern nets
- **SGD + momentum**: strong for CNNs on vision tasks
- Consider weight decay (L2 regularization), and consider excluding biases/norm layers from weight decay.

### Learning rate schedules
Schedules often matter as much as optimizer choice:
- step decay
- cosine decay
- warmup + cosine (common for Transformers)

In [ ]:

import torch.optim as optim

# Example optimizer with parameter groups (common: no weight decay for bias/norm)
def param_groups_weight_decay(model, weight_decay=0.01):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: 
            continue
        if p.ndim == 1 or name.endswith(".bias"):  # biases & norm weights
            no_decay.append(p)
        else:
            decay.append(p)
    return [
        {"params": decay, "weight_decay": weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]

m = TinyMLP().to(device)
opt = optim.AdamW(param_groups_weight_decay(m, 0.01), lr=3e-4)
print("param groups:", len(opt.param_groups), [pg["weight_decay"] for pg in opt.param_groups])

## 3.2 Regularization and stabilization

Key tools:
- **Weight decay** (AdamW preferred)
- **Dropout**: helps reduce co-adaptation
- **BatchNorm/LayerNorm**: normalization for stability
- **Label smoothing**: improves calibration and generalization
- **Early stopping**: stop when validation metric stops improving
- **Gradient clipping**: prevents exploding gradients (esp. RNNs, Transformers)

In [ ]:

import torch.nn as nn

class RegularizedMLP(nn.Module):
    def __init__(self, in_dim=10, hidden=64, out_dim=3, p=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden, out_dim),
        )
    def forward(self, x):
        return self.net(x)

reg_mlp = RegularizedMLP().to(device)

## 3.3 Gradient clipping & accumulation

### Clipping
Use `torch.nn.utils.clip_grad_norm_` after `backward()` and before `step()`.

### Accumulation
If you want an effective batch size larger than your GPU memory allows:
- accumulate gradients over `k` mini-batches
- call `optimizer.step()` every `k` steps

In [ ]:

import torch.nn.utils as nn_utils

def train_one_epoch_with_accum(model, dl, opt, loss_fn, accum_steps=4, max_norm=1.0):
    model.train()
    opt.zero_grad(set_to_none=True)
    total = 0.0
    for i, (xb, yb) in enumerate(dl):
        pred = model(xb)
        loss = loss_fn(pred, yb) / accum_steps
        loss.backward()
        if (i + 1) % accum_steps == 0:
            nn_utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
            opt.step()
            opt.zero_grad(set_to_none=True)
        total += loss.item() * xb.size(0) * accum_steps
    return total / len(dl.dataset)

# quick sanity example: use regression dl from earlier
model_tmp = nn.Linear(D, 1).to(device)
opt_tmp = optim.AdamW(model_tmp.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()
print("epoch loss:", train_one_epoch_with_accum(model_tmp, dl, opt_tmp, loss_fn))

## 3.4 Mixed precision training (AMP)

On GPUs that support it, automatic mixed precision can speed up training and reduce memory.

Pattern:
- `autocast` for forward pass
- `GradScaler` for stable scaling

In [ ]:

from torch.cuda.amp import autocast, GradScaler

use_amp = torch.cuda.is_available()
scaler = GradScaler(enabled=use_amp)

model_amp = nn.Linear(D, 1).to(device)
opt_amp = optim.AdamW(model_amp.parameters(), lr=1e-2)
loss_fn = nn.MSELoss()

for step, (xb, yb) in enumerate(dl):
    with autocast(enabled=use_amp):
        pred = model_amp(xb)
        loss = loss_fn(pred, yb)
    opt_amp.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt_amp)
    scaler.update()
    if step == 5:
        break

print("AMP ran (enabled=%s)" % use_amp)

## 3.5 Debugging and introspection

### Tools you should know
- `torch.autograd.set_detect_anomaly(True)` to find NaNs/infs in backward
- forward/backward hooks to inspect activations and gradients
- `torch.nan_to_num`, gradient norm monitoring, and assert-based checks

### Hook example

In [ ]:

torch.autograd.set_detect_anomaly(False)

act_stats = {}

def save_activation(name):
    def hook(module, inp, out):
        with torch.no_grad():
            act_stats[name] = {
                "mean": out.mean().item(),
                "std": out.std().item(),
                "min": out.min().item(),
                "max": out.max().item(),
            }
    return hook

toy = TinyMLP().to(device)
toy.fc1.register_forward_hook(save_activation("fc1"))

_ = toy(torch.randn(64, 10).to(device))
act_stats

## 3.6 Profiling
Use the PyTorch profiler to understand CPU/GPU time, memory, and kernel hotspots.

In practice:
- start with `torch.utils.bottleneck`
- then use `torch.profiler` for detailed traces

In [ ]:

import torch.profiler as profiler

m = TinyMLP().to(device)
inp = torch.randn(1024, 10).to(device)

with profiler.profile(activities=[profiler.ProfilerActivity.CPU] + 
                      ([profiler.ProfilerActivity.CUDA] if torch.cuda.is_available() else []),
                      record_shapes=True) as prof:
    for _ in range(50):
        out = m(inp)

print(prof.key_averages().table(sort_by="cpu_time_total", row_limit=10))

## 3.7 CNNs (Vision): key patterns

Typical tensor shape for images:
- **NCHW**: `[batch, channels, height, width]`

Core ops:
- `Conv2d`, `MaxPool2d`, `BatchNorm2d`
- `Flatten`, `Linear`

Below is a small CNN and a toy dataset example (random data). Swap in real datasets (e.g., MNIST/CIFAR) when you are ready.

In [ ]:

class SmallCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes),
        )
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

cnn = SmallCNN().to(device)
xb = torch.randn(32, 1, 28, 28).to(device)
logits = cnn(xb)
print(logits.shape)

## 3.8 NLP basics in PyTorch: embeddings, padding, masking

For text modeling you typically need:
- **Tokenizer**: map text → token IDs
- **Vocabulary**: token ↔ integer mapping
- **Numericalization**: tokens → ids
- **Padding**: make sequences in a batch the same length
- **Attention masks** or **lengths**: tell the model which elements are padding

We'll start with a minimal word-level tokenizer and vocabulary.

In [ ]:

from collections import Counter

def simple_word_tokenize(text: str):
    # A minimal whitespace+punct split. For real work, use a robust tokenizer.
    import re
    text = text.lower().strip()
    return re.findall(r"[a-z0-9]+|[^\s\w]", text)

class Vocab:
    def __init__(self, tokens, min_freq=1, specials=("<pad>","<unk>","<bos>","<eos>")):
        counts = Counter(tokens)
        self.itos = list(specials)
        for tok, c in counts.most_common():
            if c >= min_freq and tok not in specials:
                self.itos.append(tok)
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]

    def encode(self, tokens, add_bos=False, add_eos=False):
        ids = []
        if add_bos: ids.append(self.bos_id)
        ids += [self.stoi.get(t, self.unk_id) for t in tokens]
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        return [self.itos[i] if 0 <= i < len(self.itos) else "<bad_id>" for i in ids]

corpus = [
    "Hello world!",
    "Hello PyTorch. PyTorch makes tensors and models.",
    "Tokenization turns text into tokens, then ids."
]
tokens = [t for s in corpus for t in simple_word_tokenize(s)]
vocab = Vocab(tokens, min_freq=1)
print("vocab size:", len(vocab.itos))
print("encode:", vocab.encode(simple_word_tokenize("Hello world!"), add_bos=True, add_eos=True))

### Padding and a custom `collate_fn`

A DataLoader can batch variable-length sequences using a `collate_fn`:
- encode each example
- pad to max length in the batch
- return `input_ids` and `lengths` (or masks)

In [ ]:

from torch.utils.data import Dataset, DataLoader

class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

def collate_text(batch, vocab: Vocab):
    texts, labels = zip(*batch)
    encoded = [vocab.encode(simple_word_tokenize(t), add_bos=True, add_eos=True) for t in texts]
    lengths = torch.tensor([len(e) for e in encoded], dtype=torch.long)
    max_len = int(lengths.max())
    input_ids = torch.full((len(encoded), max_len), vocab.pad_id, dtype=torch.long)
    for i, e in enumerate(encoded):
        input_ids[i, :len(e)] = torch.tensor(e, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return input_ids.to(device), lengths.to(device), labels.to(device)

texts = ["I like PyTorch.", "PyTorch is fast.", "I like models.", "Tokenizers make ids."]
labels = [1, 1, 0, 0]
tds = TextDataset(texts, labels)
tdl = DataLoader(tds, batch_size=2, shuffle=True, collate_fn=lambda b: collate_text(b, vocab))

batch = next(iter(tdl))
[input.shape for input in batch]

### A strong baseline for text classification: `EmbeddingBag`

`nn.EmbeddingBag` can pool token embeddings efficiently (sum/mean/max).
We'll use mean pooling with padded sequences.

In [ ]:

class EmbeddingBagClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_classes=2, pad_id=0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids, lengths):
        # input_ids: (B, T)
        emb = self.embed(input_ids)               # (B, T, E)
        mask = (input_ids != vocab.pad_id).unsqueeze(-1)  # (B,T,1)
        emb = emb * mask
        pooled = emb.sum(dim=1) / lengths.unsqueeze(-1)   # mean pooling
        return self.fc(pooled)

clf = EmbeddingBagClassifier(len(vocab.itos), pad_id=vocab.pad_id).to(device)
opt = optim.AdamW(clf.parameters(), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(30):
    total = 0.0
    for input_ids, lengths, y in tdl:
        logits = clf(input_ids, lengths)
        loss = loss_fn(logits, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        total += loss.item()
    if epoch % 10 == 0:
        print("epoch", epoch, "loss", total/len(tdl))

---

# 4) Advanced Course

This section covers “what you need in real systems”:
- compilation and export
- quantization
- distributed training
- large-model tactics
- deployment patterns

## 4.1 `torch.compile` (PyTorch 2.x)

`torch.compile` can speed up models by capturing and optimizing execution graphs.
It can improve performance substantially, but requires some discipline:
- avoid Python-side data-dependent control flow (or accept graph breaks)
- keep tensor shapes reasonably stable
- prefer PyTorch ops over Python loops

In [ ]:

# torch.compile is available in PyTorch 2.0+
import torch

def run_compile_demo():
    m = TinyMLP().to(device)
    x = torch.randn(2048, 10).to(device)
    m.eval()
    # compile is optional; fall back safely if not available
    if hasattr(torch, "compile"):
        cm = torch.compile(m)  # default backend/mode
        with torch.inference_mode():
            y1 = m(x)
            y2 = cm(x)
        print("compiled ok. max diff:", (y1 - y2).abs().max().item())
    else:
        print("torch.compile not available in this torch build.")

run_compile_demo()

### Performance hygiene for `torch.compile`
- Avoid creating new tensors inside tight loops (pre-allocate when possible)
- Use `torch.inference_mode()` for eval
- Keep shapes stable (dynamic shapes are supported but may reduce optimization)
- Minimize graph breaks (unsupported ops / Python side effects)

If something fails or gets slower, first try:
- `torch.compile(model, mode="reduce-overhead")`
- use fewer dynamic shapes
- profile before/after

## 4.2 `torch.export`

`torch.export` captures a **single graph** representation (an `ExportedProgram`) suitable for deployment in Python-less runtimes and tooling.
Unlike `torch.compile`, export is a “one-shot” capture and is stricter about traceability.

Use cases:
- deployment (including edge tooling such as ExecuTorch)
- ahead-of-time graph transformations
- interoperability flows

In [ ]:

# Minimal torch.export example (works on many models; may require newer versions)
import torch

def try_export():
    if not hasattr(torch, "export"):
        print("torch.export not available in this torch build.")
        return
    m = TinyMLP().to(device).eval()
    example = (torch.randn(2, 10).to(device),)
    try:
        ep = torch.export.export(m, example)
        print("ExportedProgram created.")
        print(ep)
    except Exception as e:
        print("export failed:", type(e).__name__, str(e)[:200], "...")

try_export()

## 4.3 ONNX export (overview)

ONNX export is a common interop path for inference runtimes. In PyTorch you typically use:
- `torch.onnx.export`

Caveats:
- not every PyTorch op is supported by ONNX
- dynamic shapes require extra work

In [ ]:

import torch

def try_onnx_export():
    m = TinyMLP().to("cpu").eval()
    x = torch.randn(1, 10)
    try:
        torch.onnx.export(m, x, "tiny_mlp.onnx",
                          input_names=["x"], output_names=["logits"],
                          opset_version=17)
        print("Exported ONNX to tiny_mlp.onnx")
    except Exception as e:
        print("ONNX export failed:", type(e).__name__, str(e)[:200], "...")

try_onnx_export()

## 4.4 Quantization (PTQ/QAT) — what to know

Quantization reduces model size and can speed up CPU inference by using int8 (or other low precision).
Two broad approaches:
- **PTQ** (post-training quantization): quantize after training using calibration data
- **QAT** (quantization-aware training): simulate quantization during training for better accuracy

PyTorch provides quantization tooling under `torch.ao.quantization` (and evolving newer PT2E flows).
Quantization is a deep topic; treat this as a starting map.

In [ ]:

import torch
import torch.nn as nn

# A tiny quantization-friendly model
qmodel = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2),
).eval()

# Dynamic quantization (simple for Linear/LSTM on CPU)
try:
    qdyn = torch.ao.quantization.quantize_dynamic(
        qmodel, {nn.Linear}, dtype=torch.qint8
    )
    x = torch.randn(4, 10)
    y = qdyn(x)
    print("Dynamic quantization OK. Output shape:", y.shape)
except Exception as e:
    print("Quantization not available / failed:", type(e).__name__, str(e)[:200], "...")

## 4.5 Distributed training: DDP (Data Distributed Parallel) and beyond

For multi-GPU training, DDP is the standard baseline:
- one process per GPU
- gradients are all-reduced efficiently

Launch typically with:
```bash
torchrun --nproc_per_node=NUM_GPUS train.py
```

Beyond DDP:
- **FSDP** (Fully Sharded Data Parallel) shards parameters/gradients/optimizer state for large models
- pipeline + tensor parallelism for very large models

This notebook can’t fully run DDP inside one cell in a portable way, but here is a minimal *script template* you can paste into `train_ddp.py`.

In [ ]:

ddp_template = r'''
# train_ddp.py
import os
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

def main():
    dist.init_process_group(backend="nccl")  # "gloo" for CPU
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)

    # toy data
    X = torch.randn(4096, 10)
    y = torch.randint(0, 2, (4096,))
    ds = TensorDataset(X, y)
    sampler = DistributedSampler(ds, shuffle=True)
    dl = DataLoader(ds, batch_size=128, sampler=sampler, num_workers=2, pin_memory=True)

    model = nn.Sequential(nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 2)).to(device)
    model = DDP(model, device_ids=[local_rank])
    opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(5):
        sampler.set_epoch(epoch)
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            logits = model(xb)
            loss = loss_fn(logits, yb)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()

    dist.destroy_process_group()

if __name__ == "__main__":
    main()
'''
print(ddp_template[:800] + "\n...\n")

## 4.6 Large-model training tactics (practical checklist)

When models get large, most “failures” are engineering issues:
- OOM (out of memory)
- slow input pipeline
- unstable training
- poor checkpointing
- nondeterministic bugs

### Memory tactics
- AMP mixed precision
- gradient accumulation
- activation checkpointing (recompute activations during backward)
- FSDP / sharding
- use smaller sequence length and curriculum it upward

### Throughput tactics
- pinned memory + `non_blocking=True`
- `num_workers` tuning
- avoid Python work in the training step
- consider `torch.compile` for steady-shape workloads

### Reliability tactics
- checkpoint everything (model/opt/sched/scaler/RNG)
- validate frequently
- log grad norms and loss stats

## 4.7 Tokenizers: Beginner → Advanced (with PyTorch integration)

### Why tokenization matters
Tokenizers define:
- what a “token” is (word, char, subword, byte)
- how text maps to IDs
- what special tokens exist (`<pad>`, `<bos>`, `<eos>`, `<unk>`)
- how sequences are truncated/padded
- what masks are required (attention masks, causal masks)

### Three common levels
1. **Character tokenization**: simplest; large sequences
2. **Word tokenization**: compact; unknown words are an issue
3. **Subword (BPE/WordPiece/Unigram)**: standard for modern NLP/LLMs

Below we build:
- a character tokenizer (pure Python)
- a tiny BPE tokenizer (educational; not a full industrial implementation)

In [ ]:

import re
from collections import defaultdict

class CharTokenizer:
    def __init__(self, texts, specials=("<pad>","<unk>","<bos>","<eos>")):
        chars = set()
        for t in texts:
            chars.update(list(t))
        self.itos = list(specials) + sorted(chars)
        self.stoi = {c:i for i,c in enumerate(self.itos)}
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]

    def encode(self, text, add_bos=False, add_eos=False):
        ids = []
        if add_bos: ids.append(self.bos_id)
        ids += [self.stoi.get(c, self.unk_id) for c in text]
        if add_eos: ids.append(self.eos_id)
        return ids

    def decode(self, ids):
        return "".join(self.itos[i] if 0 <= i < len(self.itos) else "�" for i in ids)

ctok = CharTokenizer(corpus)
print("char vocab:", len(ctok.itos))
print(ctok.encode("Hello!", add_bos=True, add_eos=True))

### Minimal BPE tokenizer (educational)

This is a small BPE implementation for learning:
- starts from characters within words
- iteratively merges the most frequent adjacent pair
- results in a subword vocabulary

Important:
- industrial tokenizers handle normalization, byte-level tricks, unicode edge cases, caching, and efficient training/inference
- for production, consider specialized libraries (e.g., Hugging Face tokenizers) and then feed IDs into PyTorch

Still, understanding BPE helps you reason about models and data.

In [ ]:

def bpe_get_stats(words):
    stats = Counter()
    for w, freq in words.items():
        symbols = w.split()
        for i in range(len(symbols) - 1):
            stats[(symbols[i], symbols[i+1])] += freq
    return stats

def bpe_merge_pair(pair, words):
    a, b = pair
    pattern = re.compile(rf'(?<!\S){re.escape(a)}\s+{re.escape(b)}(?!\S)')
    merged = {}
    for w, freq in words.items():
        merged_w = pattern.sub(a + b, w)
        merged[merged_w] = freq
    return merged

class BPETokenizer:
    def __init__(self, vocab, merges, specials=("<pad>","<unk>","<bos>","<eos>")):
        self.specials = list(specials)
        self.itos = self.specials + sorted(vocab - set(self.specials))
        self.stoi = {t:i for i,t in enumerate(self.itos)}
        self.merges = merges
        self.pad_id = self.stoi["<pad>"]
        self.unk_id = self.stoi["<unk>"]
        self.bos_id = self.stoi["<bos>"]
        self.eos_id = self.stoi["<eos>"]

    @staticmethod
    def train(texts, num_merges=100, min_freq=2):
        # Build word frequency
        counts = Counter()
        for text in texts:
            for tok in simple_word_tokenize(text):
                if tok.strip():
                    counts[tok] += 1

        # Initialize "words" as space-separated characters + end marker
        words = {}
        for w, f in counts.items():
            if f >= min_freq:
                words[" ".join(list(w)) + " </w>"] = f

        merges = []
        for _ in range(num_merges):
            stats = bpe_get_stats(words)
            if not stats:
                break
            best = max(stats, key=stats.get)
            merges.append(best)
            words = bpe_merge_pair(best, words)

        # Build vocab
        vocab = set()
        for w in words.keys():
            vocab.update(w.split())
        vocab.add("</w>")
        return BPETokenizer(vocab=vocab, merges=merges)

    def encode_word(self, word):
        # Start from chars + end marker, then apply merges greedily
        symbols = list(word) + ["</w>"]
        # Apply merges in training order
        for a, b in self.merges:
            i = 0
            new = []
            while i < len(symbols):
                if i < len(symbols)-1 and symbols[i] == a and symbols[i+1] == b:
                    new.append(a+b)
                    i += 2
                else:
                    new.append(symbols[i])
                    i += 1
            symbols = new
        return symbols

    def encode(self, text, add_bos=False, add_eos=False):
        tokens = []
        if add_bos: tokens.append("<bos>")
        for w in simple_word_tokenize(text):
            tokens.extend(self.encode_word(w))
        if add_eos: tokens.append("<eos>")
        ids = [self.stoi.get(t, self.unk_id) for t in tokens]
        return ids

    def decode(self, ids):
        toks = [self.itos[i] for i in ids]
        # naive decode: join and strip end markers
        out = []
        for t in toks:
            if t in self.specials:
                continue
            if t.endswith("</w>"):
                out.append(t[:-4])
            else:
                out.append(t)
        return "".join(out)

bpe = BPETokenizer.train(corpus, num_merges=50, min_freq=1)
print("BPE vocab size:", len(bpe.itos))
print("encode:", bpe.encode("Hello PyTorch!", add_bos=True, add_eos=True)[:30])

### Using a tokenizer in a PyTorch language model

A language model predicts the next token.
Common shapes:
- input_ids: `[batch, time]`
- logits: `[batch, time, vocab_size]`

We'll build a tiny **decoder-only Transformer** (educational) on a tiny corpus. It will not be “good”, but it demonstrates:
- embeddings
- positional encoding
- causal masking
- cross-entropy next-token loss

In [ ]:

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, D)
        return x + self.pe[:, :x.size(1)]

class TinyDecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model=128, nhead=4, num_layers=2, dim_ff=256, max_len=256, pad_id=0):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = PositionalEncoding(d_model, max_len=max_len)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=dim_ff, batch_first=True)
        self.tr = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids):
        x = self.tok_emb(input_ids)
        x = self.pos(x)
        T = input_ids.size(1)
        causal = torch.triu(torch.ones(T, T, device=input_ids.device), diagonal=1).bool()
        x = self.tr(x, mask=causal)
        return self.lm_head(x)

# Build a tiny dataset for next-token prediction using BPE
def make_lm_data(texts, tokenizer, block_size=32):
    ids = []
    for t in texts:
        ids += tokenizer.encode(t, add_bos=True, add_eos=True)
    ids = torch.tensor(ids, dtype=torch.long)
    # Create (x, y) pairs
    xs, ys = [], []
    for i in range(0, len(ids) - block_size - 1, block_size):
        x = ids[i:i+block_size]
        y = ids[i+1:i+block_size+1]
        xs.append(x); ys.append(y)
    return torch.stack(xs), torch.stack(ys)

lm_x, lm_y = make_lm_data(corpus * 50, bpe, block_size=32)
lm_ds = torch.utils.data.TensorDataset(lm_x, lm_y)
lm_dl = torch.utils.data.DataLoader(lm_ds, batch_size=16, shuffle=True)

lm = TinyDecoderLM(vocab_size=len(bpe.itos), pad_id=bpe.pad_id, max_len=64).to(device)
opt = torch.optim.AdamW(lm.parameters(), lr=3e-4)

lm.train()
for epoch in range(5):
    total = 0.0
    for xb, yb in lm_dl:
        xb = xb.to(device); yb = yb.to(device)
        logits = lm(xb)                       # (B,T,V)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), yb.view(-1))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        total += loss.item()
    print("epoch", epoch, "loss", total/len(lm_dl))

### Sampling from the tiny LM (greedy sampling)

In [ ]:

@torch.inference_mode()
def generate(model, tokenizer, prompt, max_new_tokens=40):
    model.eval()
    ids = tokenizer.encode(prompt, add_bos=True)
    x = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]  # last token
        next_id = torch.argmax(logits, dim=-1).item()
        x = torch.cat([x, torch.tensor([[next_id]], device=device)], dim=1)
        # stop if eos
        if tokenizer.itos[next_id] == "<eos>":
            break
    return x.squeeze(0).tolist()

gen_ids = generate(lm, bpe, "Hello")
print("token ids:", gen_ids[:50])
print("decoded :", bpe.decode(gen_ids))

---

# 5) Practical Templates (Copy/Paste)

These templates are meant to be reused in your own projects.

## 5.1 A robust training step template
- supports AMP
- supports grad clipping
- supports accumulation
- logs loss and grad norm

Copy into `trainer.py` in your project.

In [ ]:

trainer_template = r'''
import torch
from torch.cuda.amp import autocast, GradScaler
from torch.nn.utils import clip_grad_norm_

class Trainer:
    def __init__(self, model, optimizer, loss_fn, device,
                 use_amp=True, max_grad_norm=None, accum_steps=1):
        self.model = model
        self.optimizer = optimizer
        self.loss_fn = loss_fn
        self.device = device
        self.use_amp = use_amp and torch.cuda.is_available()
        self.scaler = GradScaler(enabled=self.use_amp)
        self.max_grad_norm = max_grad_norm
        self.accum_steps = accum_steps
        self._step_in_accum = 0

    def train_step(self, batch):
        self.model.train()
        xb, yb = batch
        xb = xb.to(self.device, non_blocking=True)
        yb = yb.to(self.device, non_blocking=True)

        with autocast(enabled=self.use_amp):
            logits = self.model(xb)
            loss = self.loss_fn(logits, yb) / self.accum_steps

        self.scaler.scale(loss).backward()
        self._step_in_accum += 1

        grad_norm = None
        if self._step_in_accum == self.accum_steps:
            if self.max_grad_norm is not None:
                self.scaler.unscale_(self.optimizer)
                grad_norm = clip_grad_norm_(self.model.parameters(), self.max_grad_norm)

            self.scaler.step(self.optimizer)
            self.scaler.update()
            self.optimizer.zero_grad(set_to_none=True)
            self._step_in_accum = 0

        return {"loss": loss.item() * self.accum_steps, "grad_norm": None if grad_norm is None else float(grad_norm)}
'''
print(trainer_template[:800] + "\n...\n")

## 5.2 A tokenization + DataLoader template

This shows the *production shape* of a text pipeline:
- tokenizer.encode(text) -> list[int]
- Dataset returns ids
- collate pads and builds attention masks

In [ ]:

text_pipeline_template = r'''
import torch
from torch.utils.data import Dataset, DataLoader

class TextClsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts, self.labels, self.tokenizer = texts, labels, tokenizer
    def __len__(self):
        return len(self.texts)
    def __getitem__(self, idx):
        ids = self.tokenizer.encode(self.texts[idx], add_bos=True, add_eos=True)
        return ids, self.labels[idx]

def collate_pad(batch, pad_id=0):
    ids_list, labels = zip(*batch)
    lengths = torch.tensor([len(x) for x in ids_list], dtype=torch.long)
    max_len = int(lengths.max())
    input_ids = torch.full((len(ids_list), max_len), pad_id, dtype=torch.long)
    attn_mask = torch.zeros((len(ids_list), max_len), dtype=torch.bool)
    for i, ids in enumerate(ids_list):
        input_ids[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)
        attn_mask[i, :len(ids)] = True
    labels = torch.tensor(labels, dtype=torch.long)
    return input_ids, attn_mask, labels
'''
print(text_pipeline_template[:900] + "\n...\n")

---

# 6) Reference Cheat Sheets

## 6.1 Common tensor shapes
- tabular: `[B, D]`
- sequences: `[B, T, D]` (batch-first) or `[T, B, D]` (legacy)
- images (PyTorch conv): `[B, C, H, W]`

## 6.2 Classification with `CrossEntropyLoss`
- model outputs **logits**: shape `[B, num_classes]`
- labels are integer class IDs: shape `[B]`
- do **not** apply softmax yourself (loss does it internally)

## 6.3 Mixed precision
- forward in autocast
- backward through GradScaler
- unscale before grad clipping

## 6.4 Saving checkpoints (recommended keys)
- `model_state`
- `optimizer_state`
- `scheduler_state` (if any)
- `scaler_state` (if AMP)
- `epoch`, `global_step`, `best_metric`
- RNG states (`torch`, `cuda`, `numpy`, `random`) if you need exact reproducibility

---

# 7) Next Steps

If you want to go deeper, you typically specialize into one or more tracks:

- **Vision:** torchvision models, augmentations, detection/segmentation, mixed precision, compile
- **NLP/LLMs:** robust subword tokenizers, Transformer training at scale, distributed + FSDP, inference optimization
- **Production:** export/quantize, deployment (server + edge), monitoring, testing, reproducibility

This notebook is designed to be both a course and a long-term reference.

---

# References (selected)
- PyTorch install & start page: https://pytorch.org/
- `torch.compile` tutorial: https://docs.pytorch.org/tutorials/intermediate/torch_compile_tutorial.html
- `torch.export` guide: https://docs.pytorch.org/docs/stable/user_guide/torch_compiler/export.html
- `torch.export` tutorial: https://docs.pytorch.org/tutorials/intermediate/torch_export_tutorial.html
- ExecuTorch docs: https://docs.pytorch.org/executorch/
- TorchServe maintenance notice: https://docs.pytorch.org/serve/

# Appendix A: Training/Evaluation Patterns (Production Hygiene)

Most issues in real training are not “model architecture”; they are:
- data bugs (label leakage, misaligned preprocessing)
- evaluation bugs (wrong metric, wrong averaging, accidentally evaluating on train)
- unstable optimization (LR too high, missing warmup, exploding grads)
- checkpoint bugs (saving only the model, losing optimizer/scheduler/scaler)

This appendix gives copy/paste patterns that cover these failure modes.

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F

def accuracy_from_logits(logits, y_true):
    preds = logits.argmax(dim=-1)
    return (preds == y_true).float().mean().item()

@torch.inference_mode()
def evaluate(model, dl, loss_fn, device):
    model.eval()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for xb, yb in dl:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_acc  += accuracy_from_logits(logits, yb) * bs
        n += bs
    return {"loss": total_loss / n, "acc": total_acc / n}

def train_epoch(model, dl, opt, loss_fn, device, max_grad_norm=None):
    model.train()
    total_loss, total_acc, n = 0.0, 0.0, 0
    for xb, yb in dl:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        if max_grad_norm is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
        opt.step()
        bs = xb.size(0)
        total_loss += loss.item() * bs
        total_acc  += accuracy_from_logits(logits, yb) * bs
        n += bs
    return {"loss": total_loss / n, "acc": total_acc / n}

## Early stopping and checkpointing (simple)

Early stopping is best implemented around **a validation metric**.
Store the "best so far" checkpoint separately.

Below is a simple template for a supervised classification job.

In [ ]:

early_stopping_template = r'''
best_val = float("inf")
patience = 5
bad_epochs = 0

for epoch in range(num_epochs):
    train_stats = train_epoch(model, train_dl, opt, loss_fn, device, max_grad_norm=1.0)
    val_stats   = evaluate(model, val_dl, loss_fn, device)

    if val_stats["loss"] < best_val:
        best_val = val_stats["loss"]
        bad_epochs = 0
        torch.save({
            "model_state": model.state_dict(),
            "opt_state": opt.state_dict(),
            "epoch": epoch,
            "best_val": best_val,
        }, "best.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= patience:
            print(f"Early stopping at epoch {epoch} (best_val={best_val:.4f})")
            break
'''
print(early_stopping_template)

# Appendix B: Transfer Learning (Vision)

Transfer learning is often the strongest baseline in vision:
- start with a pretrained CNN/ViT
- replace the classifier head
- optionally freeze the backbone then unfreeze later

This requires `torchvision`. If unavailable in your environment, install it:
```bash
pip install torchvision
```

In [ ]:

transfer_learning_template = r'''
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example: ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
# Replace final layer
model.fc = nn.Linear(model.fc.in_features, num_classes)
model.to(device)

# Freeze backbone (optional)
for name, p in model.named_parameters():
    if not name.startswith("fc."):
        p.requires_grad = False

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=3e-4)
'''
print(transfer_learning_template[:900] + "\n...\n")

# Appendix C: RNN/LSTM/GRU in PyTorch (When you still need them)

Transformers dominate many tasks, but RNNs remain useful:
- small models on tiny devices
- low-latency streaming
- when you want a strong baseline quickly

Key shapes:
- `nn.RNN/LSTM/GRU` default: `(seq, batch, features)` unless `batch_first=True`.

In [ ]:

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden=128, num_classes=2, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(embed_dim, hidden, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden*2, num_classes)

    def forward(self, input_ids, lengths):
        x = self.emb(input_ids)
        # pack padded for efficiency
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, (h, c) = self.lstm(packed)
        # h: (num_layers*directions, B, hidden)
        h_last = torch.cat([h[-2], h[-1]], dim=-1)
        return self.fc(h_last)

# quick forward check
lstm_clf = LSTMClassifier(len(vocab.itos), pad_id=vocab.pad_id).to(device)
input_ids, lengths, y = next(iter(tdl))
logits = lstm_clf(input_ids, lengths)
logits.shape

# Appendix D: Transformer Building Blocks in PyTorch

PyTorch provides:
- `nn.MultiheadAttention` (attention primitive)
- `nn.TransformerEncoderLayer` / `nn.TransformerDecoderLayer` (high-level blocks)

Important details:
- Provide **attention masks** correctly (causal and padding)
- Watch shapes: batch-first vs sequence-first

In [ ]:

mha = nn.MultiheadAttention(embed_dim=64, num_heads=4, batch_first=True).to(device)
x = torch.randn(2, 8, 64).to(device)  # (B,T,D)

# causal mask: block attending to future positions
T = x.size(1)
causal_mask = torch.triu(torch.ones(T, T, device=device), diagonal=1).bool()

attn_out, attn_weights = mha(x, x, x, attn_mask=causal_mask)
print(attn_out.shape, attn_weights.shape)

# Appendix E: `torch.func` (vmap/grad) for advanced differentiation

Modern PyTorch integrates many “functorch” capabilities under `torch.func`.
These are useful for:
- per-sample gradients
- Jacobians/Hessians
- vectorizing model evaluations

Not every environment ships with full support, but the concept is important.

In [ ]:

# torch.func is available in modern PyTorch (2.x)
import torch
from torch import func

def f(params, x):
    W, b = params
    return (x @ W + b).sum()

W = torch.randn(5, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
x = torch.randn(10, 5)

# grad w.r.t. params
g = func.grad(f)( (W, b), x )
print("dW shape:", g[0].shape, "db shape:", g[1].shape)

# Appendix F: Activation checkpointing (recompute) to save memory

Activation checkpointing trades compute for memory:
- store fewer intermediate activations
- recompute them during backward

Use `torch.utils.checkpoint.checkpoint` on large submodules.

In [ ]:

from torch.utils.checkpoint import checkpoint

class CheckpointedBlock(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(d, d*4),
            nn.GELU(),
            nn.Linear(d*4, d),
        )
    def forward(self, x):
        # checkpoint expects a function; module call works too
        return checkpoint(self.block, x)

blk = CheckpointedBlock().to(device)
x = torch.randn(8, 256, device=device, requires_grad=True)
y = blk(x).sum()
y.backward()
print("ok (grad):", x.grad.abs().mean().item())

# Appendix G: `torch.compile` debugging (graph breaks)

If compiled performance is not improving, or compilation fails, you often need to find:
- unsupported operations
- Python-side control flow
- dynamic shapes exploding the graph

Useful tools (availability may depend on version):
- `torch._dynamo.explain(model, *args)`
- `TORCH_LOGS="+dynamo"` environment variable

In [ ]:

compile_debug_template = r'''
import torch

model = ...
example = ...

compiled = torch.compile(model)

# Explaining graph breaks (if available)
import torch._dynamo as dynamo
explanation = dynamo.explain(model, *example)
print(explanation)

# Logging (bash):
# export TORCH_LOGS="+dynamo"
# python train.py
'''
print(compile_debug_template)

# Appendix H: TorchServe note (serving)

TorchServe remains usable, but its documentation notes it is in “limited maintenance”.
For new projects, consider:
- serving with a thin Python API layer (FastAPI/Flask) + `torch.inference_mode()`
- exporting and running in specialized runtimes (e.g., ExecuTorch for edge)

The best choice depends on your constraints and org tooling.

In [ ]:

serving_template = r'''
# server.py (FastAPI sketch)
# pip install fastapi uvicorn
import torch
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()
model = ...  # load your model
model.eval()

class Inp(BaseModel):
    x: list[float]

@app.post("/predict")
def predict(inp: Inp):
    with torch.inference_mode():
        x = torch.tensor(inp.x).float().unsqueeze(0)
        logits = model(x)
        probs = torch.softmax(logits, dim=-1).squeeze(0).tolist()
    return {"probs": probs}
'''
print(serving_template[:900] + "\n...\n")

# Appendix I: Hugging Face tokenizers integration (optional, recommended for production NLP)

If you use a Hugging Face tokenizer:
- tokenize in Python (fast Rust tokenizer)
- return tensors to PyTorch
- use attention masks and labels in your training loop

This is the common, production-grade approach for modern NLP and LLM fine-tuning.

In [ ]:

hf_template = r'''
# pip install transformers tokenizers
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

batch = tokenizer(
    ["hello pytorch", "tokenizers produce ids"],
    padding=True, truncation=True, return_tensors="pt"
)
input_ids = batch["input_ids"]         # (B,T)
attention_mask = batch["attention_mask"]  # (B,T)
'''
print(hf_template)